## Step 0 — Verify GPU and environment

In [ ]:
import subprocess
import os

# Check GPU
gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
    capture_output=True,
    text=True
)

print(
    gpu_check.stdout
    if gpu_check.returncode == 0
    else "No GPU detected — enable GPU in Settings before continuing."
)

# Check Kaggle input files
print("\nContents of /kaggle/input/")

if os.path.exists("/kaggle/input"):
    for item in os.listdir("/kaggle/input"):
        print(" ", item)
else:
    print("  (none found — attach your dataset via 'Add Input' first)")

## Step 1 — Install pinned dependencies

Installed once, no reinstall later in the notebook. The original notebook
uninstalled and reinstalled these same packages a second time in a later
cell (for the now-removed diffusers alternative pipeline) — that redundancy
is gone here.

**After this cell finishes, restart the session** (Kaggle: Run → Restart
Session), then continue from Step 2. This matches the original notebook's
requirement and is still necessary — some of these packages don't take
effect in the same running kernel that installed them.

In [ ]:
import subprocess, sys
python = sys.executable

print("Installing pinned dependencies for kohya_ss compatibility...")

subprocess.run([python, "-m", "pip", "uninstall", "-y",
    "diffusers", "transformers", "huggingface-hub", "accelerate", "peft"])

subprocess.run([python, "-m", "pip", "install", "-q",
    "diffusers==0.25.1",
    "transformers==4.38.2",
    "huggingface-hub==0.21.4",
    "accelerate==0.27.2",
    "peft==0.9.0",
    "safetensors>=0.4.0",
    "bitsandbytes>=0.41.0",
    "xformers",
    "einops",
    "omegaconf",
    "toml",
    "ftfy",
    "albumentations",
    "timm",
    "voluptuous",
])

print("\nDone. NOW RESTART THE SESSION (Run -> Restart Session).")
print("After restarting, skip this cell and continue from Step 2.")

## Step 2 — Clone kohya_ss (sd-scripts)

In [19]:
import os, subprocess

KOHYA_DIR = "/kaggle/working/kohya_ss"

if os.path.exists(KOHYA_DIR):
    subprocess.run(["rm", "-rf", KOHYA_DIR])
    print("Removed old kohya_ss clone")

print("Cloning kohya_ss...")
result = subprocess.run([
    "git", "clone", "-q", "--depth", "1", "-b", "v23.1.6",
    "https://github.com/kohya-ss/sd-scripts", KOHYA_DIR
])

if result.returncode != 0:
    print("Tagged version failed, cloning main...")
    subprocess.run(["git", "clone", "-q", "https://github.com/kohya-ss/sd-scripts", KOHYA_DIR])

trainer = os.path.join(KOHYA_DIR, "sdxl_train_network.py")
if os.path.exists(trainer):
    print(f"Trainer found: {trainer}")
else:
    print(f"Trainer NOT found at {trainer} -- check the clone above")

Cloning kohya_ss...


fatal: Remote branch v23.1.6 not found in upstream origin


Tagged version failed, cloning main...
Trainer found: /kaggle/working/kohya_ss/sdxl_train_network.py


## Step 2b — Install kohya_ss requirements and verify imports

In [20]:
import subprocess, sys, os

python = sys.executable
KOHYA_DIR = "/kaggle/working/kohya_ss"
req_file = os.path.join(KOHYA_DIR, "requirements.txt")

if os.path.exists(req_file):
    print("Installing kohya_ss requirements...")
    subprocess.run([python, "-m", "pip", "install", "-q", "-r", req_file])
else:
    print("requirements.txt not found -- skipping (already installed in Step 1)")

print("\nVerifying imports...")
try:
    import diffusers, transformers, accelerate
    print(f"  diffusers   : {diffusers.__version__}")
    print(f"  transformers: {transformers.__version__}")
    print(f"  accelerate  : {accelerate.__version__}")
    print("All imports OK")
except ImportError as e:
    print(f"Import error: {e}")
    print("Restart the session and re-run from Step 1.")

Installing kohya_ss requirements...

Verifying imports...
  diffusers   : 0.25.1
  transformers: 4.38.2
  accelerate  : 0.27.2
All imports OK


ERROR: file:///kaggle/working (from -r /kaggle/working/kohya_ss/requirements.txt (line 49)) does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## Step 3 — Download SDXL base model

Pulled directly from Hugging Face rather than uploaded as a Kaggle Dataset
-- reproducible without a manual multi-GB upload step, and cached so
re-running this cell doesn't re-download.

In [21]:
from huggingface_hub import hf_hub_download
import shutil, os

MODEL_LOCAL_PATH = "/kaggle/working/sd_xl_base_1.0.safetensors"

if not os.path.exists(MODEL_LOCAL_PATH):
    print("Downloading SDXL base model (this can take a while)...")
    downloaded_path = hf_hub_download(
        repo_id="stabilityai/stable-diffusion-xl-base-1.0",
        filename="sd_xl_base_1.0.safetensors",
    )
    shutil.copy(downloaded_path, MODEL_LOCAL_PATH)
    print(f"Model saved to {MODEL_LOCAL_PATH}")
else:
    print(f"Model already present at {MODEL_LOCAL_PATH}")

Model already present at /kaggle/working/sd_xl_base_1.0.safetensors


## Step 4 — Configure run and verify dataset

Set `RUN = "a"` for clean images or `RUN = "b"` for cloaked images, then
run this cell. **This is the single place that controls which dataset gets
trained** -- the training cell in Step 6 reads these variables directly,
rather than hardcoding its own copy of them (which is what the original
notebook's Run A / Run B cells did, making the `RUN` toggle here have no
actual effect on them).

Change `RUN` and re-run this cell, then re-run Step 6, to switch between
Run A and Run B -- no need to duplicate the training cell.

In [22]:
import os

RUN = "a"   # "a" for run-a0-1, change to "b" if you attach run-b

# ============================================================
# KAGGLE PATHS
# ============================================================

INPUT_BASE = "/kaggle/input/datasets/leothiii"

DATASET_PATH = os.path.join(
    INPUT_BASE,
    "runa06"
)

MODEL_PATH = "/kaggle/working/sd_xl_base_1.0.safetensors"

OUTPUT_DIR = f"/kaggle/working/outputs/lora_run_{RUN}"
OUTPUT_NAME = f"lora_run_{RUN}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# CHECK
# ============================================================

print(f"RUN        : {RUN.upper()}")
print(f"Dataset    : {DATASET_PATH}")
print(f"Output dir : {OUTPUT_DIR}")

if os.path.exists(DATASET_PATH):
    print("\n✅ Dataset path exists!")

    print("\nContents:")
    for item in os.listdir(DATASET_PATH):
        print(" ", item)
else:
    print("\n❌ Dataset path NOT FOUND")

RUN        : A
Dataset    : /kaggle/input/datasets/leothiii/runa06
Output dir : /kaggle/working/outputs/lora_run_a

✅ Dataset path exists!

Contents:
  10_person


##### Step 5 — Run LoRA training


In [ ]:
# ============================================================
# SDXL LoRA — FINAL STABLE KAGGLE TRAINING CELL
# Tesla T4 / 14.6 GB VRAM
# ============================================================

import os
import shuti












































































































































































































































































































l
from pathlib import Path

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

KOHYA_DIR = "/kaggle/working/kohya_ss"
TRAIN_SCRIPT = f"{KOHYA_DIR}/sdxl_train_network.py"

MODEL_PATH = "/kaggle/working/sd_xl_base_1.0.safetensors"

# Original read-only dataset
SOURCE_DATASET = "/kaggle/input/datasets/leothiii/runa06"

# Writable copy used for training
TRAIN_DATA_DIR = "/kaggle/working/runa06_train"

# Final LoRA output
OUTPUT_DIR = "/kaggle/working/outputs/lora_run_a_final"
OUTPUT_NAME = "lora_runa06"

# ------------------------------------------------------------
# 2. BASIC CHECKS
# ------------------------------------------------------------

print("=" * 70)
print("CHECKING TRAINING ENVIRONMENT")
print("=" * 70)

assert os.path.exists(KOHYA_DIR), f"Kohya directory not found: {KOHYA_DIR}"
assert os.path.exists(TRAIN_SCRIPT), f"Training script not found: {TRAIN_SCRIPT}"
assert os.path.exists(MODEL_PATH), f"SDXL model not found: {MODEL_PATH}"
assert os.path.exists(SOURCE_DATASET), f"Dataset not found: {SOURCE_DATASET}"

print("Kohya:", TRAIN_SCRIPT)
print("Model:", MODEL_PATH)
print("Source dataset:", SOURCE_DATASET)

# ------------------------------------------------------------
# 3. COPY DATASET TO /kaggle/working
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREPARING WRITABLE TRAINING DATASET")
print("=" * 70)

if os.path.exists(TRAIN_DATA_DIR):
    print("Removing previous working dataset...")
    shutil.rmtree(TRAIN_DATA_DIR)

shutil.copytree(SOURCE_DATASET, TRAIN_DATA_DIR)

print("Dataset copied to:")
print(TRAIN_DATA_DIR)


print("\n" + "=" * 70)
print("FIXING CAPTIONS")
print("=" * 70)

caption_files = list(Path(TRAIN_DATA_DIR).rglob("*.txt"))

print(f"Found {len(caption_files)} caption files.")

for txt_file in caption_files:
    original = txt_file.read_text(encoding="utf-8").strip()

    # Normalize the captions for this character LoRA.
    #
    # Handles the current caption:
    # "a photo of ohwx person"
    #
    # and also avoids accidentally duplicating the trigger.
    if "ohwx" in original.lower():
        new_caption = "ohwx person"
    else:
        # If a caption somehow does not contain ohwx,
        # still give the image the intended trigger caption.
        new_caption = "ohwx person"

    txt_file.write_text(new_caption, encoding="utf-8")

print("Captions normalized to:")
print("  ohwx person")

# Show a few examples
print("\nCaption samples:")
for txt_file in caption_files[:5]:
    print(f"  {txt_file.name}: {txt_file.read_text(encoding='utf-8').strip()}")

# ------------------------------------------------------------
# 5. CREATE OUTPUT DIRECTORY
# ------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 6. VERIFY DATASET COUNTS
# ------------------------------------------------------------

image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

image_files = [
    p for p in Path(TRAIN_DATA_DIR).rglob("*")
    if p.suffix.lower() in image_extensions
]

caption_files = list(Path(TRAIN_DATA_DIR).rglob("*.txt"))

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"Images:   {len(image_files)}")
print(f"Captions: {len(caption_files)}")

assert len(image_files) == 30, (
    f"Expected 30 images based on the validated dataset, "
    f"but found {len(image_files)}"
)

assert len(caption_files) == 30, (
    f"Expected 30 captions based on the validated dataset, "
    f"but found {len(caption_files)}"
)

# ------------------------------------------------------------
# 7. GPU CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GPU")
print("=" * 70)

os.system("nvidia-smi --query-gpu=name,memory.total --format=csv")

# ------------------------------------------------------------
# 8. FINAL TRAINING COMMAND
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING FINAL SDXL LoRA TRAINING")
print("=" * 70)

command = f"""
/usr/bin/python3 -u -m accelerate.commands.launch \
  --num_processes=1 \
  --num_machines=1 \
  --mixed_precision=fp16 \
  {TRAIN_SCRIPT} \
  --pretrained_model_name_or_path={MODEL_PATH} \
  --train_data_dir={TRAIN_DATA_DIR} \
  --output_dir={OUTPUT_DIR} \
  --output_name={OUTPUT_NAME} \
  --save_model_as=safetensors \
  --resolution=512,512 \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --train_batch_size=1 \
  --max_train_epochs=10 \
  --learning_rate=1e-5 \
  --unet_lr=1e-5 \
  --text_encoder_lr=1e-6 \
  --optimizer_type=AdamW8bit \
  --mixed_precision=fp16 \
  --save_precision=fp16 \
  --no_half_vae \
  --gradient_checkpointing \
  --sdpa \
  --cache_latents \
  --vae_batch_size=1 \
  --caption_extension=.txt \
  --shuffle_caption \
  --keep_tokens=1 \
  --save_every_n_epochs=1 \
  --seed=42
"""

print(command)

# ------------------------------------------------------------
# 9. RUN TRAINING
# ------------------------------------------------------------

exit_code = os.system(command)

# ------------------------------------------------------------
# 10. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING PROCESS FINISHED")
print("=" * 70)

print("Exit code:", exit_code)
print("Output directory:", OUTPUT_DIR)

if exit_code == 0:
    print("\nSUCCESS.")
    print("Your LoRA files should be in:")
    print(OUTPUT_DIR)

    print("\nSaved files:")
    for p in sorted(Path(OUTPUT_DIR).glob("*")):
        print(" ", p.name)
else:
    print("\nTraining exited with a non-zero status.")
    print("Check the training log above for the exact error.")

CHECKING TRAINING ENVIRONMENT
Kohya: /kaggle/working/kohya_ss/sdxl_train_network.py
Model: /kaggle/working/sd_xl_base_1.0.safetensors
Source dataset: /kaggle/input/datasets/leothiii/runa06

PREPARING WRITABLE TRAINING DATASET
Dataset copied to:
/kaggle/working/runa06_train

FIXING CAPTIONS
Found 30 caption files.
Captions normalized to:
  ohwx person

Caption samples:
  img3.txt: ohwx person
  img19.txt: ohwx person
  img24.txt: ohwx person
  img6.txt: ohwx person
  img29.txt: ohwx person

DATASET SUMMARY
Images:   30
Captions: 30

GPU
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

STARTING FINAL SDXL LoRA TRAINING

/usr/bin/python3 -u -m accelerate.commands.launch   --num_processes=1   --num_machines=1   --mixed_precision=fp16   /kaggle/working/kohya_ss/sdxl_train_network.py   --pretrained_model_name_or_path=/kaggle/working/sd_xl_base_1.0.safetensors   --train_data_dir=/kaggle/working/runa06_train   --output_dir=/kaggle/working/outputs/lora_run_a_final   --output_na

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node`

accelerator device: cuda:0


                    INFO     loading U-Net from           sdxl_model_util.py:209
                             checkpoint                                         
2026-09-15 10:31:01 INFO     U-Net: <All keys matched     sdxl_model_util.py:211
                             successfully>                                      
                    INFO     building text encoders       sdxl_model_util.py:214
                    INFO     loading text encoders from   sdxl_model_util.py:267
                             checkpoint                                         
2026-09-15 10:31:02 INFO     text encoder 1: <All keys    sdxl_model_util.py:281
                             matched successfully>                              
2026-09-15 10:31:08 INFO     text encoder 2: <All keys    sdxl_model_util.py:285
                             matched successfully>                              
                    INFO     building VAE                 sdxl_model_util.py:288
                    INFO    

import network module: networks.lora


...

2026-09-15 10:31:38 INFO     use 8-bit AdamW optimizer | {}     optimizer.py:106


override steps. steps for 10 epochs is / 指定エポックまでのステップ数: 3000
running training / 学習開始
  num train images * repeats / 学習画像の数×繰り返し回数: 300
  num validation images * repeats / 学習画像の数×繰り返し回数: 0
  num reg images / 正則化画像の数: 0
  num batches per epoch / 1epochのバッチ数: 300
  num epochs / epoch数: 10
  batch size per device / バッチサイズ: 1
  gradient accumulation steps / 勾配を合計するステップ数 = 1
  total optimization steps / 学習ステップ数: 3000


/kaggle/working/kohya_ss/library/sdxl_lpw_stable_diffusion.py:82: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
2026-09-15 10:32:01 INFO     unet dtype: torch.float16,    train_network.py:1519
                             device: cuda:0                                     
                    INFO     text_encoder [0] dtype:       train_network.py:1525
                             torch.float16, device: cuda:0                      
                    INFO     text_encoder [1] dtype:       train_network.py:1525
                             torch.float16, device: cuda:0                      
steps:   0%|          | 0/3000 [00:00<?, ?it/s]


epoch 1/10



2026-09-15 10:32:02 INFO     epoch is incremented. current_epoch: dataset.py:464
                             0, epoch: 1                                        
2026-09-15 10:32:02 INFO     epoch is incremented. current_epoch: dataset.py:464
                             0, epoch: 1                                        
2026-09-15 10:32:02 INFO     epoch is incremented. current_epoch: dataset.py:464
                             0, epoch: 1                                        
2026-09-15 10:32:02 INFO     epoch is incremented. current_epoch: dataset.py:464
                             0, epoch: 1                                        
steps:   2%|▏         | 55/3000 [01:35<1:25:03,  1.73s/it, avr_loss=0.152]